## Intro
The goal of this notebook is use different machine learning algorithms to perform binary classification for malware detection.

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import tensorflow as tf
import pickle
from scipy.special import boxcox

from sklearn.model_selection import train_test_split, cross_val_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from keras import callbacks

In [3]:
import sys
import unittest
import import_ipynb
import malwarefunctions

class TestRunModel(unittest.TestCase):
    def test_df_shape(self):
        file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
        df = pd.read_csv(file_path, sep='|')
        self.assertEqual(df.shape, (138047, 57))
    def test_value_counts(self):
        file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
        df = pd.read_csv(file_path, sep='|')
        self.assertEqual(df['legitimate'].value_counts()[0], 96724)
        self.assertEqual(df['legitimate'].value_counts()[1], 41323)
    def test_normalization(self):
        file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
        df = pd.read_csv(file_path, sep='|')
        df = malwarefunctions.preprocess(df)
        X = df.drop(columns=['legitimate'])
        for label, content in X.describe().items():
            if label != "legitimate":
                self.assertAlmostEqual(content['mean'], 0)
                self.assertAlmostEqual(content['std'], 1)
    def test_components_drop(self):
        file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
        df = pd.read_csv(file_path, sep='|')
        df = malwarefunctions.preprocess(df)
        self.assertEqual(df.shape, (138047, 37))
    def test_gbc_training(self):
        file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
        df = pd.read_csv(file_path, sep='|')
        df = malwarefunctions.preprocess(df)
        X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
        model = GradientBoostingClassifier(random_state=42)
        model.fit(X_train, y_train)
        self.assertIsNotNone(model)
        self.assertTrue(hasattr(model, 'feature_importances_'))
    def test_gbc_prediction(self):
        file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
        df = pd.read_csv(file_path, sep='|')
        df = malwarefunctions.preprocess(df)
        X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
        model = GradientBoostingClassifier(random_state=42)
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        self.assertEqual(len(predictions), len(X_test))
    def test_gbc_accuracy(self):
        file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
        df = pd.read_csv(file_path, sep='|')
        df = malwarefunctions.preprocess(df)
        X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
        model = GradientBoostingClassifier(random_state=42)
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)
        self.assertGreater(accuracy, 0.9)
    def test_nb_training(self):
        file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
        df = pd.read_csv(file_path, sep='|')
        df = pd.read_csv(file_path, sep='|')
        df = malwarefunctions.preprocess(df)
        X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
        model = GaussianNB()
        model.fit(X_train, y_train)
        self.assertIsNotNone(model)
        self.assertTrue(hasattr(model, 'class_prior_'))
    def test_nb_prediction(self):
        file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
        df = pd.read_csv(file_path, sep='|')
        df = malwarefunctions.preprocess(df)
        X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
        model = GaussianNB()
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        self.assertEqual(len(predictions), len(X_test))
    def test_nb_accuracy(self):
        file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
        df = pd.read_csv(file_path, sep='|')
        df = malwarefunctions.preprocess(df)
        X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
        model = GaussianNB()
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)
        self.assertGreater(accuracy, 0.9)
    def test_ann_training(self):
        file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
        df = pd.read_csv(file_path, sep='|')
        df = malwarefunctions.preprocess(df)
        X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
        input_shape = [X_train.shape[1]]
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(units=64, activation='relu', input_shape=input_shape),
            tf.keras.layers.Dense(units=64, activation='relu'),
            tf.keras.layers.Dense(units=1)
        ])
        model.build()
        model.compile(optimizer='adam', loss='mae', metrics=['accuracy'])  
        earlystopping = callbacks.EarlyStopping(monitor="val_loss",
                                                    mode="min",
                                                    patience=5,
                                                    restore_best_weights=True)
        history = model.fit(X_train, y_train, validation_data=(X_test,y_test), batch_size=256, epochs=60,callbacks=[earlystopping], verbose=0)
        self.assertIsNotNone(model)
    def test_ann_prediction(self):
        file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
        df = pd.read_csv(file_path, sep='|')
        df = malwarefunctions.preprocess(df)
        X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
        input_shape = [X_train.shape[1]]
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(units=64, activation='relu', input_shape=input_shape),
            tf.keras.layers.Dense(units=64, activation='relu'),
            tf.keras.layers.Dense(units=1)
        ])
        model.build()
        model.compile(optimizer='adam', loss='mae', metrics=['accuracy'])  
        earlystopping = callbacks.EarlyStopping(monitor="val_loss",
                                                    mode="min",
                                                    patience=5,
                                                    restore_best_weights=True)
        history = model.fit(X_train, y_train, validation_data=(X_test,y_test), batch_size=256, epochs=60,callbacks=[earlystopping], verbose=0)
        predictions = (model.predict(X_test) > 0.5).astype("int32")
        self.assertEqual(len(predictions), len(X_test))
    def test_ann_accuracy(self):
        file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
        df = pd.read_csv(file_path, sep='|')
        df = malwarefunctions.preprocess(df)
        X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
        input_shape = [X_train.shape[1]]
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(units=64, activation='relu', input_shape=input_shape),
            tf.keras.layers.Dense(units=64, activation='relu'),
            tf.keras.layers.Dense(units=1)
        ])
        model.build()
        model.compile(optimizer='adam', loss='mae', metrics=['accuracy'])  
        earlystopping = callbacks.EarlyStopping(monitor="val_loss",
                                                    mode="min",
                                                    patience=5,
                                                    restore_best_weights=True)
        history = model.fit(X_train, y_train, validation_data=(X_test,y_test), batch_size=256, epochs=60,callbacks=[earlystopping], verbose=0)
        self.assertGreater(history.history['accuracy'][-1], 0.9)
    def test_svm_training(self):
        file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
        df = pd.read_csv(file_path, sep='|')
        df = malwarefunctions.preprocess(df)
        X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
        model = SVC(random_state=42)
        model.fit(X_train, y_train)
        self.assertIsNotNone(model)
        self.assertTrue(hasattr(model, 'class_weight_'))
    def test_svm_prediction(self):
        file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
        df = pd.read_csv(file_path, sep='|')
        df = malwarefunctions.preprocess(df)
        X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
        model = SVC(random_state=42)
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        self.assertEqual(len(predictions), len(X_test))
    def test_svm_accuracy(self):
        file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
        df = pd.read_csv(file_path, sep='|')
        df = malwarefunctions.preprocess(df)
        X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
        model = SVC(random_state=42)
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)
        self.assertGreater(accuracy, 0.9)
    def test_logreg_training(self):
        file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
        df = pd.read_csv(file_path, sep='|')
        df = malwarefunctions.preprocess(df)
        X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
        model = LogisticRegression(random_state=42, solver='sag')
        model.fit(X_train, y_train)
        self.assertIsNotNone(model)
        self.assertTrue(hasattr(model, 'intercept_'))
    def test_logreg_prediction(self):
        file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
        df = pd.read_csv(file_path, sep='|')
        df = malwarefunctions.preprocess(df)
        X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
        model = LogisticRegression(random_state=42, solver='sag')
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        self.assertEqual(len(predictions), len(X_test))
    def test_logreg_accuracy(self):
        file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
        df = pd.read_csv(file_path, sep='|')
        df = malwarefunctions.preprocess(df)
        X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
        model = LogisticRegression(random_state=42, solver='sag')
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)
        self.assertGreater(accuracy, 0.9)
    def test_rfc_training(self):
        file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
        df = pd.read_csv(file_path, sep='|')
        df = malwarefunctions.preprocess(df)
        X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
        model = RandomForestClassifier(random_state=42)
        model.fit(X_train, y_train)
        self.assertIsNotNone(model)
        self.assertTrue(hasattr(model, 'max_depth'))
    def test_rfc_prediction(self):
        file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
        df = pd.read_csv(file_path, sep='|')
        df = malwarefunctions.preprocess(df)
        X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
        model = RandomForestClassifier(random_state=42)
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        self.assertEqual(len(predictions), len(X_test))
    def test_rfc_accuracy(self):
        file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
        df = pd.read_csv(file_path, sep='|')
        df = malwarefunctions.preprocess(df)
        X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
        model = RandomForestClassifier(random_state=42)
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)
        self.assertGreater(accuracy, 0.9)
    def test_dtc_training(self):
        file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
        df = pd.read_csv(file_path, sep='|')
        df = malwarefunctions.preprocess(df)
        X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
        model = DecisionTreeClassifier(random_state=42)
        model.fit(X_train, y_train)
        self.assertIsNotNone(model)
        self.assertTrue(hasattr(model, 'max_depth'))
    def test_dtc_prediction(self):
        file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
        df = pd.read_csv(file_path, sep='|')
        df = malwarefunctions.preprocess(df)
        X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
        model = DecisionTreeClassifier(random_state=42)
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        self.assertEqual(len(predictions), len(X_test))
    def test_dtc_accuracy(self):
        file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
        df = pd.read_csv(file_path, sep='|')
        df = malwarefunctions.preprocess(df)
        X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
        model = DecisionTreeClassifier(random_state=42)
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)
        self.assertGreater(accuracy, 0.9)
        
# suite = unittest.TestLoader().loadTestsFromTestCase(TestRunModel)
# unittest.TextTestRunner(verbosity=4,stream=sys.stderr).run(suite)

In [28]:
def test_preprocess():
    class TestRunModel(unittest.TestCase):
        def test_df_shape(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            self.assertEqual(df.shape, (138047, 57))
        def test_value_counts(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            self.assertEqual(df['legitimate'].value_counts()[0], 96724)
            self.assertEqual(df['legitimate'].value_counts()[1], 41323)
        def test_normalization(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X = df.drop(columns=['legitimate'])
            for label, content in X.describe().items():
                if label != "legitimate":
                    self.assertAlmostEqual(content['mean'], 0)
                    self.assertAlmostEqual(content['std'], 1)
        def test_components_drop(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            self.assertEqual(df.shape, (138047, 37))
        
    suite = unittest.TestLoader().loadTestsFromTestCase(TestRunModel)
    unittest.TextTestRunner(verbosity=4,stream=sys.stderr).run(suite)

In [29]:
def test_gbc():
    class TestRunModel(unittest.TestCase):
        def test_gbc_training(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = GradientBoostingClassifier(random_state=42)
            model.fit(X_train, y_train)
            self.assertIsNotNone(model)
            self.assertTrue(hasattr(model, 'feature_importances_'))
        def test_gbc_prediction(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = GradientBoostingClassifier(random_state=42)
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            self.assertEqual(len(predictions), len(X_test))
        def test_gbc_accuracy(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = GradientBoostingClassifier(random_state=42)
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            accuracy = accuracy_score(y_test, predictions)
            f1 = f1_score(y_test, predictions)
            self.assertGreaterEqual(accuracy, 0.9)
            self.assertGreaterEqual(f1, 0.9)

    suite = unittest.TestLoader().loadTestsFromTestCase(TestRunModel)
    unittest.TextTestRunner(verbosity=4,stream=sys.stderr).run(suite)

In [30]:
def test_nb():
    class TestRunModel(unittest.TestCase):
        def test_nb_training(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = GaussianNB()
            model.fit(X_train, y_train)
            self.assertIsNotNone(model)
            self.assertTrue(hasattr(model, 'class_prior_'))
        def test_nb_prediction(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = GaussianNB()
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            self.assertEqual(len(predictions), len(X_test))
        def test_nb_accuracy(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = GaussianNB()
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            accuracy = accuracy_score(y_test, predictions)
            f1 = f1_score(y_test, predictions)
            self.assertGreaterEqual(accuracy, 0.9)
            self.assertGreaterEqual(f1, 0.9)

    suite = unittest.TestLoader().loadTestsFromTestCase(TestRunModel)
    unittest.TextTestRunner(verbosity=4,stream=sys.stderr).run(suite)

In [31]:
def test_ann():
    import os
    os.environ['CUDA_VISIBLE_DEVICES'] = "0"
    
    class TestRunModel(unittest.TestCase):
        def test_ann_training(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            input_shape = [X_train.shape[1]]
            model = tf.keras.Sequential([
                tf.keras.layers.Dense(units=64, activation='relu', input_shape=input_shape),
                tf.keras.layers.Dense(units=64, activation='relu'),
                tf.keras.layers.Dense(units=1)
            ])
            model.build()
            model.compile(optimizer='adam', loss='mae', metrics=['accuracy'])  
            earlystopping = callbacks.EarlyStopping(monitor="val_loss",
                                                        mode="min",
                                                        patience=5,
                                                        restore_best_weights=True)
            history = model.fit(X_train, y_train, validation_data=(X_test,y_test), batch_size=256, epochs=60,callbacks=[earlystopping], verbose=0)
            self.assertIsNotNone(model)
        def test_ann_prediction(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            input_shape = [X_train.shape[1]]
            model = tf.keras.Sequential([
                tf.keras.layers.Dense(units=64, activation='relu', input_shape=input_shape),
                tf.keras.layers.Dense(units=64, activation='relu'),
                tf.keras.layers.Dense(units=1)
            ])
            model.build()
            model.compile(optimizer='adam', loss='mae', metrics=['accuracy'])  
            earlystopping = callbacks.EarlyStopping(monitor="val_loss",
                                                        mode="min",
                                                        patience=5,
                                                        restore_best_weights=True)
            history = model.fit(X_train, y_train, validation_data=(X_test,y_test), batch_size=256, epochs=60,callbacks=[earlystopping], verbose=0)
            predictions = (model.predict(X_test) > 0.5).astype("int32")
            self.assertEqual(len(predictions), len(X_test))
        def test_ann_accuracy(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            input_shape = [X_train.shape[1]]
            model = tf.keras.Sequential([
                tf.keras.layers.Dense(units=64, activation='relu', input_shape=input_shape),
                tf.keras.layers.Dense(units=64, activation='relu'),
                tf.keras.layers.Dense(units=1)
            ])
            model.build()
            model.compile(optimizer='adam', loss='mae', metrics=['accuracy'])  
            earlystopping = callbacks.EarlyStopping(monitor="val_loss",
                                                        mode="min",
                                                        patience=5,
                                                        restore_best_weights=True)
            history = model.fit(X_train, y_train, validation_data=(X_test,y_test), batch_size=256, epochs=60,callbacks=[earlystopping], verbose=0)
            self.assertGreater(history.history['accuracy'][-1], 0.9)

    suite = unittest.TestLoader().loadTestsFromTestCase(TestRunModel)
    unittest.TextTestRunner(verbosity=4,stream=sys.stderr).run(suite)

In [32]:
def test_svm():
    class TestRunModel(unittest.TestCase):
        def test_svm_training(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = SVC(random_state=42)
            model.fit(X_train, y_train)
            self.assertIsNotNone(model)
            self.assertTrue(hasattr(model, 'class_weight_'))
        def test_svm_prediction(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = SVC(random_state=42)
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            self.assertEqual(len(predictions), len(X_test))
        def test_svm_accuracy(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = SVC(random_state=42)
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            accuracy = accuracy_score(y_test, predictions)
            f1 = f1_score(y_test, predictions)
            self.assertGreaterEqual(accuracy, 0.9)
            self.assertGreaterEqual(f1, 0.9)

    suite = unittest.TestLoader().loadTestsFromTestCase(TestRunModel)
    unittest.TextTestRunner(verbosity=4,stream=sys.stderr).run(suite)

In [33]:
def test_logreg():
    class TestRunModel(unittest.TestCase):
        def test_logreg_training(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = LogisticRegression(random_state=42, solver='sag')
            model.fit(X_train, y_train)
            self.assertIsNotNone(model)
            self.assertTrue(hasattr(model, 'intercept_'))
        def test_logreg_prediction(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = LogisticRegression(random_state=42, solver='sag')
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            self.assertEqual(len(predictions), len(X_test))
        def test_logreg_accuracy(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = LogisticRegression(random_state=42, solver='sag')
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            accuracy = accuracy_score(y_test, predictions)
            f1 = f1_score(y_test, predictions)
            self.assertGreaterEqual(accuracy, 0.9)
            self.assertGreaterEqual(f1, 0.9)

    suite = unittest.TestLoader().loadTestsFromTestCase(TestRunModel)
    unittest.TextTestRunner(verbosity=4,stream=sys.stderr).run(suite)

In [34]:
def test_rfc():
    class TestRunModel(unittest.TestCase):
        def test_rfc_training(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = RandomForestClassifier(random_state=42)
            model.fit(X_train, y_train)
            self.assertIsNotNone(model)
            self.assertTrue(hasattr(model, 'max_depth'))
        def test_rfc_prediction(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = RandomForestClassifier(random_state=42)
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            self.assertEqual(len(predictions), len(X_test))
        def test_rfc_accuracy(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = RandomForestClassifier(random_state=42)
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            accuracy = accuracy_score(y_test, predictions)
            f1 = f1_score(y_test, predictions)
            self.assertGreaterEqual(accuracy, 0.9)
            self.assertGreaterEqual(f1, 0.9)

    suite = unittest.TestLoader().loadTestsFromTestCase(TestRunModel)
    unittest.TextTestRunner(verbosity=4,stream=sys.stderr).run(suite)

In [35]:
def test_dtc():
    class TestRunModel(unittest.TestCase):
        def test_dtc_training(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = DecisionTreeClassifier(random_state=42)
            model.fit(X_train, y_train)
            self.assertIsNotNone(model)
            self.assertTrue(hasattr(model, 'max_depth'))
        def test_dtc_prediction(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = DecisionTreeClassifier(random_state=42)
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            self.assertEqual(len(predictions), len(X_test))
        def test_dtc_accuracy(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = DecisionTreeClassifier(random_state=42)
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            accuracy = accuracy_score(y_test, predictions)
            f1 = f1_score(y_test, predictions)
            self.assertGreaterEqual(accuracy, 0.9)
            self.assertGreaterEqual(f1, 0.9)

    suite = unittest.TestLoader().loadTestsFromTestCase(TestRunModel)
    unittest.TextTestRunner(verbosity=4,stream=sys.stderr).run(suite)

In [36]:
test_preprocess()
test_gbc()
test_nb()
test_ann()
test_svm()
test_logreg()
test_rfc()
test_dtc()

test_components_drop (__main__.test_preprocess.<locals>.TestRunModel) ... ok
test_df_shape (__main__.test_preprocess.<locals>.TestRunModel) ... ok
test_normalization (__main__.test_preprocess.<locals>.TestRunModel) ... ok
test_value_counts (__main__.test_preprocess.<locals>.TestRunModel) ... ok

----------------------------------------------------------------------
Ran 4 tests in 7.564s

OK
test_gbc_accuracy (__main__.test_gbc.<locals>.TestRunModel) ... ok
test_gbc_prediction (__main__.test_gbc.<locals>.TestRunModel) ... ok
test_gbc_training (__main__.test_gbc.<locals>.TestRunModel) ... ok

----------------------------------------------------------------------
Ran 3 tests in 119.626s

OK
test_nb_accuracy (__main__.test_nb.<locals>.TestRunModel) ... ok
test_nb_prediction (__main__.test_nb.<locals>.TestRunModel) ... ok
test_nb_training (__main__.test_nb.<locals>.TestRunModel) ... ok

----------------------------------------------------------------------
Ran 3 tests in 8.025s

OK
test_ann

863/863 [==============================] - 1s 981us/step


ok
test_ann_training (__main__.test_ann.<locals>.TestRunModel) ... ok

----------------------------------------------------------------------
Ran 3 tests in 94.743s

OK
test_svm_accuracy (__main__.test_svm.<locals>.TestRunModel) ... ok
test_svm_prediction (__main__.test_svm.<locals>.TestRunModel) ... ok
test_svm_training (__main__.test_svm.<locals>.TestRunModel) ... ok

----------------------------------------------------------------------
Ran 3 tests in 185.902s

OK
test_logreg_accuracy (__main__.test_logreg.<locals>.TestRunModel) ... /usr/local/lib/python3.8/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
ok
test_logreg_prediction (__main__.test_logreg.<locals>.TestRunModel) ... /usr/local/lib/python3.8/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
ok
test_logreg_training (__main__.t